In [6]:
import os
import numpy as np
import pandas as pd

EMBEDDING_DIR = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/embeddings/keyframe_clips"
COLLECTION_NAME = "vbs25_clips"
ID_MAPPING = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/id_mapping/keyframes.csv"


In [3]:
from pymilvus import connections, utility, MilvusException, Collection
connections.connect(host="localhost", port="19530")
try:
    collections = utility.list_collections()
    print("List of collections: ", collections)
except MilvusException as e:
    print(e)


from pymilvus import MilvusClient, DataType
CLUSTER_ENDPOINT = "http://localhost:19530"
TOKEN = "root:Milvus"
client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

List of collections:  ['vbs25_clips']


In [4]:
collection = Collection('vbs25_clips')
collection.num_entities
client.get_collection_stats(collection_name="vbs25_clips")


{'row_count': 6464}

In [29]:
if not os.path.exists(ID_MAPPING):
    df = pd.DataFrame(columns=["keyframe_name", "keyframe_id"])
    df.to_csv(ID_MAPPING, index=False)
    print(f"Created {ID_MAPPING}")

Created /home/pc/LSC24_SemanticSearchWebApp/backend/data/id_mapping/keyframes.csv


In [30]:
id_mapping = pd.read_csv(ID_MAPPING)
id_mapping.set_index("keyframe_name", inplace=True)
CURR_ID = len(id_mapping)
print("Current ID: ", CURR_ID)

Current ID:  0


In [31]:
def get_current_id():
    global CURR_ID
    temp = CURR_ID
    CURR_ID += 1
    return temp

def get_keyframe_order_from_name(name):
    return int(name.split('_')[1])

def get_keyframe_name_from_entry_name(name):
    return f"V3C/{name[4:9]}/{int(name.split('_')[1]):05d}"

id_mapping_updates = []

for entry in sorted(os.scandir(EMBEDDING_DIR), key=lambda e: e.name)[:1]:
    if entry.is_dir():
        clip_name = entry.name
        clip_dir = os.path.join(EMBEDDING_DIR, clip_name)
        for clip_entry in sorted(os.scandir(clip_dir), key=lambda e: get_keyframe_order_from_name(e.name)):
            if clip_entry.is_file() and clip_entry.name.endswith('.npy'):
                name = clip_entry.name
                path = os.path.join(clip_dir, name)
                keyframe_id = get_current_id()
                keyframe_name = get_keyframe_name_from_entry_name(name)
                embedding = np.load(path).astype(np.float32)

                # Insert data to Milvus
                data = [{
                    "keyframe_id": keyframe_id,
                    "keyframe_name": keyframe_name,
                    "embedding": embedding,
                }]
                # collection.insert(data=data)
                res = client.insert(collection_name=COLLECTION_NAME, data=data)
                print(res)

                # Update ID mapping
                mapping_data = {"keyframe_name": keyframe_name, "keyframe_id": keyframe_id}
                id_mapping_updates.append(mapping_data)
                print(f"Added {keyframe_name} with ID {keyframe_id} to mapping")

            else:
                print("Not a file")

# append new data to id_mapping
updates_df = pd.DataFrame(id_mapping_updates)
updates_df.set_index("keyframe_name", inplace=True)
id_mapping = pd.concat([id_mapping, updates_df])
id_mapping.to_csv(ID_MAPPING, index=True)

{'insert_count': 1, 'ids': [0], 'cost': 0}
Added V3C/00001/00001 with ID 0 to mapping
{'insert_count': 1, 'ids': [1], 'cost': 0}
Added V3C/00001/00002 with ID 1 to mapping
{'insert_count': 1, 'ids': [2], 'cost': 0}
Added V3C/00001/00003 with ID 2 to mapping
{'insert_count': 1, 'ids': [3], 'cost': 0}
Added V3C/00001/00004 with ID 3 to mapping
{'insert_count': 1, 'ids': [4], 'cost': 0}
Added V3C/00001/00005 with ID 4 to mapping
{'insert_count': 1, 'ids': [5], 'cost': 0}
Added V3C/00001/00006 with ID 5 to mapping
{'insert_count': 1, 'ids': [6], 'cost': 0}
Added V3C/00001/00007 with ID 6 to mapping
{'insert_count': 1, 'ids': [7], 'cost': 0}
Added V3C/00001/00008 with ID 7 to mapping
{'insert_count': 1, 'ids': [8], 'cost': 0}
Added V3C/00001/00009 with ID 8 to mapping
{'insert_count': 1, 'ids': [9], 'cost': 0}
Added V3C/00001/00010 with ID 9 to mapping
{'insert_count': 1, 'ids': [10], 'cost': 0}
Added V3C/00001/00011 with ID 10 to mapping
{'insert_count': 1, 'ids': [11], 'cost': 0}
Added V3

In [5]:
# client.get(collection_name="vbs25_clips", ids=[1, 2, 3])[0]['embedding'].__len__()
client.get(collection_name="vbs25_clips", ids=[1, 2, 3])

data: ["{'keyframe_id': 1, 'keyframe_name': 'V3C/00001/00002', 'embedding': [-0.025427341, 0.07560492, 0.023208752, 0.005344784, -0.016898738, 0.007973957, 0.005550076, -0.06021886, 0.036246568, 0.02313672, -0.027040862, 0.026378166, -0.021508794, 0.008442165, 0.03650588, -0.03394154, -0.0061263326, -0.01293697, -0.030368745, -0.017921593, -0.03192464, 0.041807447, -0.01701399, 0.06091037, 0.027833216, 0.023367222, 0.0042210827, -0.014356003, -0.0013803156, -0.30933478, 0.0030289511, 0.003165812, -0.008406149, -0.015429283, -0.023079094, -0.07877434, 0.013664495, -0.041202378, -0.038666848, -0.016711455, -0.027746776, 0.002150159, 0.01782075, 0.007138384, 0.0533614, 0.022445211, 0.011251419, -0.015544534, 0.020197809, -0.059181597, -0.011734034, -0.021652859, -0.00090085185, 0.0057049445, -0.002784042, 0.024130763, 0.01712924, -0.023612132, 0.0023554508, 0.031463634, 0.02737221, 0.0009274137, 0.027919654, 0.0029875326, 0.007404903, -0.00031739156, 0.08678431, -0.00057580683, -0.0245773